# **Text Summarizer**

In [13]:
!pip uninstall -y transformers
!pip install -q transformers==4.57.3 torch sentencepiece

Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3


In [14]:
import transformers

print(transformers.__version__)

4.57.3


## Import the pipeline

In [15]:
from transformers import pipeline

## Load the Hugging Face model

In [16]:
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6"
)

Device set to use cuda:0


## Provide sample text

In [17]:
text = """
Artificial Intelligence is transforming the way people work and communicate.
It enables computers to perform tasks that traditionally require human intelligence.
These tasks include understanding language, recognising images, analysing data,
and making predictions.

Machine Learning is a subset of Artificial Intelligence that allows computers
to learn from data. Instead of being explicitly programmed for every task,
machine learning models identify patterns and improve their performance through
experience.

Deep Learning is a specialised area of machine learning that uses artificial
neural networks. It is widely used in speech recognition, image processing,
natural language processing, and autonomous systems.

Today, AI is used in healthcare, education, banking, transportation,
manufacturing, and many other industries.
"""

## Generate summary

In [18]:
result = summarizer(
    text,
    max_new_tokens=100,
    min_length=20,
    do_sample=False
)

print("Original Text:")
print(text)

print("\nGenerated Summary:")
print(result[0]["summary_text"])

Original Text:

Artificial Intelligence is transforming the way people work and communicate.
It enables computers to perform tasks that traditionally require human intelligence.
These tasks include understanding language, recognising images, analysing data,
and making predictions.

Machine Learning is a subset of Artificial Intelligence that allows computers
to learn from data. Instead of being explicitly programmed for every task,
machine learning models identify patterns and improve their performance through
experience.

Deep Learning is a specialised area of machine learning that uses artificial
neural networks. It is widely used in speech recognition, image processing,
natural language processing, and autonomous systems.

Today, AI is used in healthcare, education, banking, transportation,
manufacturing, and many other industries.


Generated Summary:
 Machine Learning is a subset of Artificial Intelligence that allows computers to learn from data . It is widely used in speech reco

## Calculate text length

In [19]:
print("Original word count:", len(text.split()))

summary = result[0]["summary_text"]

print("Summary word count:", len(summary.split()))

Original word count: 110
Summary word count: 32


In [20]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Text input
text_input = widgets.Textarea(
    value="",
    placeholder="Paste your text here...",
    description="Input:",
    layout=widgets.Layout(width="100%", height="250px")
)

# Summary length selection
length_dropdown = widgets.Dropdown(
    options=[
        ("Short", (20, 50)),
        ("Medium", (50, 100)),
        ("Long", (100, 150))
    ],
    value=(50, 100),
    description="Length:"
)

# Buttons
generate_button = widgets.Button(
    description="Generate Summary",
    button_style="success",
    icon="check"
)

clear_button = widgets.Button(
    description="Clear All",
    button_style="warning",
    icon="trash"
)

# Output area
output = widgets.Output()

def generate_summary(button):

    with output:
        clear_output()

        input_text = text_input.value.strip()

        if not input_text:
            print("Please enter some text.")
            return

        if len(input_text.split()) < 20:
            print("Please enter at least 20 words.")
            return

        min_tokens, max_tokens = length_dropdown.value

        print("Generating summary...")

        try:
            result = summarizer(
                input_text,
                max_new_tokens=max_tokens,
                min_length=min_tokens,
                do_sample=False,
                truncation=True
            )

            generated_summary = result[0]["summary_text"]

            original_count = len(input_text.split())
            summary_count = len(generated_summary.split())

            reduction = (
                (original_count - summary_count)
                / original_count
            ) * 100

            clear_output()

            print("GENERATED SUMMARY")
            print("-" * 50)
            print(generated_summary)

            print("\nSUMMARY STATISTICS")
            print("-" * 50)
            print("Original Words:", original_count)
            print("Summary Words:", summary_count)
            print("Text Reduction:", round(reduction, 2), "%")

        except Exception as e:
            clear_output()
            print("Error:", str(e))


def clear_all(button):

    text_input.value = ""

    with output:
        clear_output()


generate_button.on_click(generate_summary)
clear_button.on_click(clear_all)

display(text_input)
display(length_dropdown)
display(widgets.HBox([generate_button, clear_button]))
display(output)

Textarea(value='', description='Input:', layout=Layout(height='250px', width='100%'), placeholder='Paste your …

Dropdown(description='Length:', index=1, options=(('Short', (20, 50)), ('Medium', (50, 100)), ('Long', (100, 1…

Output()